# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library and pandas.

### Dataset Source
This dataset is described by a Croissant schema and is available at the following URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's metadata and inspect general information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review all available record sets, fields, and their `@id`s. You can use the Dataset object to browse the schema.

In [ ]:
# List all available record sets and show their @id and name
print("Available Record Sets:")
record_sets = []
for rs in metadata.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_sets.append(rs.id)
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - Field name: {fld.name} | @id: {fld.id} | dataType: {fld.data_type}")
    print()
if not record_sets:
    print("No record sets detected in the metadata. Trying to enumerate datasets from the API or distribution.")
    # Occasionally, record_sets may be missing from the metadata object,
    # but if none, we cannot continue further record-based exploration in this notebook.

## 3. Data Extraction

Load data from each available record set into a pandas DataFrame.

*Note: Be sure to use the `@id` of the record set, fields, and columns in all function calls and dictionary accesses.*

In [ ]:
# Gather available record set @id's from the data overview step
if not record_sets:
    print("No record sets found in metadata, skipping extraction step.")
else:
    dataframes = {}
    for record_set_id in record_sets:
        print(f"Loading records from RecordSet: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")

    # Display columns of the first record set (as example)
    example_rs = record_sets[0]
    print(f"\nExample DataFrame columns for RecordSet '{example_rs}':")
    print(dataframes[example_rs].columns.tolist())
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply processing to fields using their `@id`s, such as filtering numeric fields, normalization, and grouping by a key attribute. 

In [ ]:
if not record_sets:
    print("No record sets available for EDA.")
else:
    # Use first available record set and its DataFrame
    rs_id = record_sets[0]
    df = dataframes[rs_id]

    # Identify numeric fields using the field overview
    numeric_field_id = None
    group_field_id = None
    for rs in metadata.record_sets:
        if rs.id == rs_id:
            for fld in rs.fields:
                if fld.data_type in ('schema:Integer', 'schema:Float', 'schema:Number'):
                    numeric_field_id = fld.id
                    break
            # For groupby, find a Text/categorical field
            for fld in rs.fields:
                if fld.data_type == 'schema:Text' and group_field_id is None:
                    group_field_id = fld.id
            break
    if numeric_field_id and numeric_field_id in df.columns:
        print(f"Analyzing numeric field: {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by group_field_id if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical (Text) field found for groupby in this record set.")
    else:
        print("No suitable numeric fields found in the selected record set.")

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib or pandas built-ins.

In [ ]:
import matplotlib.pyplot as plt

if not record_sets or not (numeric_field_id and numeric_field_id in df.columns):
    print("No numeric field to visualize.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id exists, show boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
    else:
        print("No suitable categorical field for grouped boxplot.")

## 6. Conclusion

In this notebook, you learned how to:
- Access and parse a Croissant schema using the `mlcroissant` library;
- Identify available record sets and fields using their `@id`s;
- Extract data to pandas DataFrames referencing all fields by their `@id`;
- Apply filtering, normalization, and grouping operations using the appropriate `@id`s;
- Visualize key numeric data fields.

**Please explore the full schema further to utilize domain-specific fields and columns of interest in your analyses.**